In [1]:
import sys
!{sys.executable} -m pip install -U langchain-google-genai
!{sys.executable} -m pip install -U python-dotenv
!{sys.executable} -m pip install -U huggingface_hub transformers torch
!{sys.executable} -m pip install -U openpyxl pandas



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: C:\Users\Ana\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: C:\Users\Ana\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: C:\Users\Ana\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: C:\Users\Ana\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from huggingface_hub import InferenceClient
import json
from typing import Dict, List, Optional
import time
import pandas as pd
import os
from pathlib import Path
from pydantic import BaseModel, Field

C:\Users\Ana\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")
judge_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.1,
)

In [4]:
class JudgeOutput(BaseModel):
    """Structured evaluation output"""
    correctness: float = Field(description="Score 0.0, 0.5, or 1.0")
    completeness: float = Field(description="Score 0.0, 0.3, 0.7, or 1.0")
    numerical_precision: float = Field(description="Score 0.0, 0.5, or 1.0")
    hallucinations: float = Field(description="Score 0.0, 0.5, or 1.0")
    clinical_risk: float = Field(description="Score 0.0, 0.5, or 1.0")
    discrepancies: List[str] = Field(default_factory=list)
    missing_information: List[str] = Field(default_factory=list)
    fabricated_information: List[str] = Field(default_factory=list)
    overall_assessment: str = Field(default="")

judge_llm_struct = judge_llm.with_structured_output(
    method="json_schema",
    schema=JudgeOutput
)

In [5]:
CLINICAL_EVALUATION_PROMPT = """You are an expert medical AI evaluator. Evaluate the AI assistant's response against the reference answer.

**Evaluation Criteria:**

1. **Correctness:**
   - 1.0 = Completely correct, matches reference answer
   - 0.5 = Partially correct, some minor errors
   - 0.0 = Incorrect

2. **Completeness:**
   - 1.0 = Covers all important information from reference answer
   - 0.7 = Misses minor details
   - 0.3 = Misses significant information
   - 0.0 = Incomplete or missing major points

3. **Numerical Precision:**
   - 1.0 = All numbers/dosages/values match reference exactly
   - 0.5 = Minor numerical discrepancies
   - 0.0 = Significant numerical errors or missing values

4. **Hallucinations:**
   - 0.0 = No fabricated information
   - 0.5 = Minor additions not in reference but not harmful
   - 1.0 = Significant fabricated or false information

5. **Clinical Risk:**
   - 0.0 = No risk, follows reference answer
   - 0.5 = Minor inefficiency risk
   - 1.0 = Significant risk to patient safety

Compare the response directly with the reference answer. Identify specific discrepancies, missing information, and fabricated content.

Return your evaluation as a JSON object with these exact fields:
- correctness (float)
- completeness (float)
- numerical_precision (float)
- hallucinations (float)
- clinical_risk (float)
- discrepancies (list of strings)
- missing_information (list of strings)
- fabricated_information (list of strings)
- overall_assessment (string)

Query: {question}
Ground truth: {reference_answer}
Generation: {response}
"""


In [6]:
def evaluate_clinical_response(
    question: str,
    response: str,
    reference_answer: str,
    retry_delay: int = 2
) -> Dict:
    """
    Evaluate medical/clinical response against reference answer
    """
    prompt = CLINICAL_EVALUATION_PROMPT.format(
        question=question,
        reference_answer=reference_answer,
        response=response
    )

    try:
        time.sleep(retry_delay)
        judge_response = judge_llm.invoke(prompt).content

        start = judge_response.find('{')
        end = judge_response.rfind('}') + 1

        if start != -1 and end != 0:
            json_str = judge_response[start:end]
            evaluation = json.loads(json_str)
        else:
            evaluation = {
                "raw_response": judge_response,
                "note": "Could not parse structured evaluation"
            }

        return evaluation

    except Exception as e:
        return {
            "error": str(e),
            "note": "Evaluation failed"
        }


In [7]:
def load_test_questions(excel_path: str, sheet_name: str = 'Sheet1') -> pd.DataFrame:
    """
    Load test questions from Excel file
    Expected columns: 'ID', 'Question', 'Answer', 'Chapter', 'Difficulty'
    """
    df = pd.read_excel(excel_path, sheet_name=sheet_name)
    df = df[df['ID'].notna()]
    df = df.reset_index(drop=True)
    return df

# Example loading test data
test_df = load_test_questions('test_questions.xlsx')
print(f"Loaded {len(test_df)} test questions")
print("\nColumns:", test_df.columns.tolist())
print("\nFirst few questions:")
print(test_df[['ID', 'Question', 'Chapter', 'Difficulty']].head())

Loaded 250 test questions

Columns: ['ID', 'Question', 'Answer', 'Chapter', 'Difficulty']

First few questions:
    ID                                           Question  \
0  1.0  What is the primary goal of ESC guidelines for...   
1  2.0  What does VA stand for in the context of ESC g...   
2  3.0  What diagnostic test is recommended as first-l...   
3  4.0  Which imaging modality is recommended when car...   
4  5.0                          What does NSVT stand for?   

                                             Chapter Difficulty  
0                                        1. Preamble       Easy  
1                                      Abbreviations       Easy  
2  5. Diagnostic evaluation of ventricular arrhyt...       Easy  
3  5. Diagnostic evaluation of ventricular arrhyt...       Easy  
4                                      Abbreviations       Easy  


In [8]:
def evaluate_finetuned_model(
    model_instance,
    test_df: pd.DataFrame,
    question_col: str = 'Question',
    answer_col: str = 'Answer',
    output_json: str = 'evaluation_results.json',
    output_csv: str = 'judge_results.csv'
) -> tuple[List[Dict], pd.DataFrame]:
    """
    Evaluate a fine-tuned model on all test questions

    Args:
        model_instance: The fine-tuned model to evaluate
        test_df: DataFrame with test questions
        question_col: Column name for questions
        answer_col: Column name for reference answers
        output_json: Where to save detailed JSON results
        output_csv: Where to save CSV results

    Returns:
        Tuple of (results list, results dataframe)
    """
    results = []
    scores = {
        'correctness': [],
        'completeness': [],
        'numerical_precision': [],
        'hallucinations': [],
        'clinical_risk': []
    }

    try:
        for idx, row in test_df.iterrows():
            question = row[question_col]
            reference_answer = row[answer_col]
            chapter = row.get('Chapter', None)
            difficulty = row.get('Difficulty', None)
            question_id = row.get('ID', idx)

            print(f"Evaluating question {idx+1}/{len(test_df)} (ID: {question_id})")

            # Get model response
            try:
                model_response = model_instance.invoke(question).content
            except Exception as e:
                model_response = f"Error: {str(e)}"

            # Prepare prompt
            prompt = CLINICAL_EVALUATION_PROMPT.format(
                question=question,
                reference_answer=reference_answer,
                response=model_response
            )

            # Get structured evaluation
            try:
                time.sleep(2)
                out: JudgeOutput = judge_llm_struct.invoke(prompt)

                # Collect scores
                scores['correctness'].append(out.correctness)
                scores['completeness'].append(out.completeness)
                scores['numerical_precision'].append(out.numerical_precision)
                scores['hallucinations'].append(out.hallucinations)
                scores['clinical_risk'].append(out.clinical_risk)

                evaluation = out.model_dump()
            except Exception as e:
                print(f"  Evaluation error: {str(e)}")
                evaluation = {
                    "error": str(e),
                    "correctness": 0,
                    "completeness": 0,
                    "numerical_precision": 0,
                    "hallucinations": 1,
                    "clinical_risk": 1,
                    "discrepancies": [],
                    "missing_information": [],
                    "fabricated_information": [],
                    "overall_assessment": "Evaluation failed"
                }

            result = {
                "question_id": question_id,
                "question": question,
                "reference_answer": reference_answer,
                "chapter": chapter,
                "difficulty": difficulty,
                "model_response": model_response,
                **evaluation
            }

            results.append(result)

            # Save incrementally to both formats
            with open(output_json, 'w', encoding='utf-8') as f:
                json.dump(results, f, indent=2, ensure_ascii=False)

            # Save CSV incrementally
            results_df = pd.DataFrame(results)
            results_df.to_csv(output_csv, index=False, encoding='utf-8')

    except KeyboardInterrupt:
        print("\n\nEvaluation interrupted by user!")
    except Exception as e:
        print(f"\n\nEvaluation stopped due to error: {str(e)}")
    finally:
        # Always save final results
        if results:
            print(f"\nSaving {len(results)} results...")
            with open(output_json, 'w', encoding='utf-8') as f:
                json.dump(results, f, indent=2, ensure_ascii=False)

            results_df = pd.DataFrame(results)
            results_df.to_csv(output_csv, index=False, encoding='utf-8')
            print(f"Results saved to {output_json} and {output_csv}")
        else:
            print("No results to save")
            results_df = pd.DataFrame()

    return results, results_df
def print_evaluation_summary(evaluation: Dict) -> None:
    """
    Print formatted evaluation summary
    """
    if 'correctness' not in evaluation:
        print(json.dumps(evaluation, indent=2, ensure_ascii=False))
        return

    print("="*80)
    print("EVALUATION SUMMARY")
    print("="*80)

    print(f"\nCorrectness: {evaluation['correctness']}")
    print(f"Completeness: {evaluation['completeness']}")
    print(f"Numerical Precision: {evaluation['numerical_precision']}")
    print(f"Hallucinations: {evaluation['hallucinations']}")
    print(f"Clinical Risk: {evaluation['clinical_risk']}")

    if 'discrepancies' in evaluation and evaluation['discrepancies']:
        print(f"\nDiscrepancies:")
        for disc in evaluation['discrepancies']:
            print(f"  • {disc}")

    if 'missing_information' in evaluation and evaluation['missing_information']:
        print(f"\nMissing Information:")
        for info in evaluation['missing_information']:
            print(f"  • {info}")

    if 'fabricated_information' in evaluation and evaluation['fabricated_information']:
        print(f"\nFabricated Information:")
        for info in evaluation['fabricated_information']:
            print(f"  • {info}")

In [9]:
def compare_models_clinical(
    question: str,
    guidelines: str,
    models: List[Dict]
) -> List[Dict]:
    """
    Compare multiple models against clinical guidelines
    """
    results = []

    for model_info in models:
        model_name = model_info["name"]
        model_instance = model_info["instance"]

        response = model_instance.invoke(question).content
        evaluation = evaluate_clinical_response(question, response, guidelines)

        results.append({
            "model": model_name,
            "response": response,
            "evaluation": evaluation
        })

    return results


In [10]:
def batch_evaluate(
    test_cases: List[Dict[str, str]],
    model_instance
) -> List[Dict]:
    """
    Evaluate model on multiple test cases

    test_cases format: [
        {
            "question": "...",
            "guidelines": "..."
        },
        ...
    ]
    """
    results = []

    for test_case in test_cases:
        response = model_instance.invoke(test_case["question"]).content
        evaluation = evaluate_clinical_response(
            test_case["question"],
            response,
            test_case["guidelines"]
        )

        results.append({
            "question": test_case["question"],
            "guidelines": test_case["guidelines"],
            "response": response,
            "evaluation": evaluation
        })

    return results

In [11]:
def calculate_aggregate_metrics(results_df: pd.DataFrame) -> Dict:
    """
    Calculate average scores from results DataFrame
    """
    metrics = ['correctness', 'completeness', 'numerical_precision', 'hallucinations', 'clinical_risk']

    aggregates = {}
    for metric in metrics:
        if metric in results_df.columns:
            values = results_df[metric].dropna()
            if len(values) > 0:
                aggregates[f'{metric}_avg'] = round(values.mean(), 4)
                aggregates[f'{metric}_min'] = round(values.min(), 4)
                aggregates[f'{metric}_max'] = round(values.max(), 4)

    return aggregates

In [12]:
EXCEL_PATH = 'test_questions.xlsx'
OUTPUT_JSON = 'evaluation_results.json'
OUTPUT_CSV = 'judge_results.csv'

test_df = load_test_questions(EXCEL_PATH)

results, results_df = evaluate_finetuned_model(
    model_instance=judge_llm,
    test_df=test_df,
    question_col='Question',
    answer_col='Answer',
    output_json=OUTPUT_JSON,
    output_csv=OUTPUT_CSV
)

aggregate_metrics = calculate_aggregate_metrics(results_df)

print("\n" + "="*80)
print("EVALUATION COMPLETE")
print("="*80)
print(f"\nTotal questions evaluated: {len(results)}")
print(f"\nResults saved to:")
print(f"  JSON: {OUTPUT_JSON}")
print(f"  CSV:  {OUTPUT_CSV}")
print("\nAggregate Metrics:")
for metric, value in aggregate_metrics.items():
    print(f"  {metric}: {value}")

# Calculate average score across all metrics
avg_correctness = aggregate_metrics.get('correctness_avg', 0)
avg_completeness = aggregate_metrics.get('completeness_avg', 0)
avg_overall = (avg_correctness + avg_completeness) / 2
print(f"\n  Overall Score: {round(avg_overall, 4)}")

Evaluating question 1/250 (ID: 1.0)
Evaluating question 2/250 (ID: 2.0)
Evaluating question 3/250 (ID: 3.0)
Evaluating question 4/250 (ID: 4.0)
Evaluating question 5/250 (ID: 5.0)
Evaluating question 6/250 (ID: 6.0)
Evaluating question 7/250 (ID: 7.0)
Evaluating question 8/250 (ID: 8.0)
Evaluating question 9/250 (ID: 9.0)
Evaluating question 10/250 (ID: 10.0)
  Evaluation error: Error calling model 'gemini-2.5-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 13.647003429s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/g